# Multi-Label Image Classification — Transfer Learning Comparison

Compares three pretrained ImageNet models with **only the last feature block + classifier head unfrozen**:
- **ResNet18** — classic residual network, fast and compact
- **EfficientNet-B0** — compound-scaled network, excellent accuracy/size trade-off
- **MobileNetV2** — depthwise-separable, extremely lightweight

All backbone layers are frozen; only the top layers are updated.

**Labels (12 classes):** `pen, paper, book, clock, phone, laptop, chair, desk, bottle, keychain, backpack, calculator`

## 1. Imports & Setup

In [ ]:
import sys
import torch
import torch.nn as nn
from pathlib import Path
from torchvision import models as tv_models

sys.path.append("../")
from eval import LABEL_ORDER
from utils import (
    set_seed, load_dataset, split_dataset, subsample_subset,
    get_train_transform, get_eval_transform, build_dataloaders,
    plot_per_class_examples, plot_multilabel_examples,
    run_baselines, print_model_info,
    train_model, save_checkpoint, load_checkpoint,
    plot_training_history, plot_multi_arch_histories,
    collect_test_predictions, categorize_predictions,
    show_prediction_examples, plot_per_class_metrics,
    plot_confusion_matrices, plot_prediction_heatmap,
    show_saliency_examples, compute_multilabel_metrics,
    NUM_LABELS, METRIC_KEYS,
)

SEED = 42
set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


## 2. Config

In [ ]:
BASE_DATA_DIR   = "../data/aggregated"
IMAGE_SIZE      = 128
BATCH_SIZE      = 128
SPLIT           = [0.8, 0.1, 0.1]

USE_SUBSET      = False
SUBSET_FRACTION = 0.1

CHECKPOINT_DIR  = Path("../checkpoints")

print(f"Labels ({NUM_LABELS}): {LABEL_ORDER}")


## 3. Data Loading & Loaders

In [ ]:
full_dataset = load_dataset(BASE_DATA_DIR)
train_raw, val_raw, test_raw = split_dataset(full_dataset, SPLIT, SEED)

if USE_SUBSET:
    train_raw = subsample_subset(train_raw, SUBSET_FRACTION, SEED)
    val_raw   = subsample_subset(val_raw,   SUBSET_FRACTION, SEED + 1)
    test_raw  = subsample_subset(test_raw,  SUBSET_FRACTION, SEED + 2)

train_transform = get_train_transform(IMAGE_SIZE)
eval_transform  = get_eval_transform(IMAGE_SIZE)
train_loader, val_loader, test_loader = build_dataloaders(
    train_raw, val_raw, test_raw, train_transform, eval_transform, BATCH_SIZE
)

print(f"Total: {len(full_dataset)}  |  "
      f"Train: {len(train_raw)}  Val: {len(val_raw)}  Test: {len(test_raw)}")


## 4. Visualize Train Samples (Per Class + Multi-Label)

In [ ]:
plot_per_class_examples(train_raw)
plot_multilabel_examples(train_raw)


## 5. Baselines

In [ ]:
run_baselines(train_loader, val_loader, test_loader, NUM_LABELS, DEVICE)


## 6. Model — Transfer Learning Architectures

**Edit this section** to add or change architectures.

`build_transfer_model(arch)` loads a pretrained backbone, freezes it, then unfreezes the last block + classifier.

In [ ]:
def build_transfer_model(arch: str, num_labels: int) -> nn.Module:
    if arch == "resnet18":
        m = tv_models.resnet18(weights=tv_models.ResNet18_Weights.IMAGENET1K_V1)
        for p in m.parameters():
            p.requires_grad = False
        for p in m.layer4.parameters():
            p.requires_grad = True
        m.fc = nn.Linear(m.fc.in_features, num_labels)

    elif arch == "efficientnet_b0":
        m = tv_models.efficientnet_b0(weights=tv_models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        for p in m.parameters():
            p.requires_grad = False
        for block in list(m.features.children())[-1:]:
            for p in block.parameters():
                p.requires_grad = True
        for p in m.classifier.parameters():
            p.requires_grad = True
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_labels)

    elif arch == "mobilenet_v2":
        m = tv_models.mobilenet_v2(weights=tv_models.MobileNet_V2_Weights.IMAGENET1K_V1)
        for p in m.parameters():
            p.requires_grad = False
        for block in list(m.features.children())[-2:]:
            for p in block.parameters():
                p.requires_grad = True
        for p in m.classifier.parameters():
            p.requires_grad = True
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_labels)

    else:
        raise ValueError(f"Unknown arch: {arch}")
    return m


ARCHS = ["resnet18", "efficientnet_b0", "mobilenet_v2"]

print(f"{'Arch':<22} {'Trainable':>10} {'Total':>10} {'Ratio':>7}  {'Size':>9}")
print("-" * 65)
for arch in ARCHS:
    m         = build_transfer_model(arch, NUM_LABELS)
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in m.parameters())
    size_mb   = total * 4 / 1024 / 1024
    size_str  = f"{size_mb:.2f} MB" if size_mb < 1024 else f"{size_mb / 1024:.3f} GB"
    print(f"{arch:<22} {trainable:>10,} {total:>10,} {100*trainable/total:>6.1f}%  {size_str:>9}")


## 7. Training — All Architectures

In [ ]:
LR           = 1e-3
WEIGHT_DECAY = 1e-3
MAX_EPOCHS   = 20

all_histories = {}
best_models   = {}
best_val_f1s  = {}

for arch in ARCHS:
    print(f"\n{'='*60}\n  {arch}\n{'='*60}")
    # Use default= to capture arch value at iteration time (avoid closure pitfall)
    best_state, best_f1, history, epochs_run = train_model(
        lambda nl, a=arch: build_transfer_model(a, nl),
        NUM_LABELS, train_loader, val_loader, DEVICE,
        lr=LR, weight_decay=WEIGHT_DECAY, max_epochs=MAX_EPOCHS,
        warmup_epochs=3,
    )
    ckpt = CHECKPOINT_DIR / f"tl_{arch}.pth"
    save_checkpoint(best_state, ckpt)

    m_final = build_transfer_model(arch, NUM_LABELS).to(DEVICE)
    m_final.load_state_dict(best_state)
    m_final.eval()

    all_histories[arch] = history
    best_models[arch]   = m_final
    best_val_f1s[arch]  = best_f1
    print(f"  Best val F1: {best_f1:.4f}")

print("\n\n=== Val F1 Summary ===")
for arch in ARCHS:
    print(f"  {arch:<22} {best_val_f1s[arch]:.4f}")


In [ ]:
plot_multi_arch_histories(all_histories, experiment_name="Transfer Learning")

# Test set metrics comparison table
print(f"\n{'Arch':<22} {'loss':>7} {'exact':>7} {'hamming':>7} "
      f"{'IoU':>7} {'prec':>7} {'rec':>7} {'F1':>7}")
print("-" * 85)
for arch, m_eval in best_models.items():
    all_l, all_p, all_pr, all_lg = [], [], [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            logits = m_eval(images)
            probs  = torch.sigmoid(logits)
            preds  = (probs >= 0.5).float()
            all_l.append(labels.cpu()); all_p.append(preds.cpu())
            all_pr.append(probs.cpu()); all_lg.append(logits.cpu())
    m = compute_multilabel_metrics(
        torch.cat(all_l), torch.cat(all_p),
        probs=torch.cat(all_pr), logits=torch.cat(all_lg),
    )
    print(f"{arch:<22} {m['loss']:>7.4f} {m['exact_match']:>7.4f} "
          f"{m['hamming_acc']:>7.4f} {m['mean_iou']:>7.4f} "
          f"{m['precision_micro']:>7.4f} {m['recall_micro']:>7.4f} {m['f1_micro']:>7.4f}")


In [ ]:
BEST_ARCH  = max(best_val_f1s, key=best_val_f1s.get)
model      = best_models[BEST_ARCH]
MODEL_PATH = CHECKPOINT_DIR / f"tl_{BEST_ARCH}.pth"
create_model = lambda nl: build_transfer_model(BEST_ARCH, nl)

print(f"Best arch: {BEST_ARCH}  (val F1={best_val_f1s[BEST_ARCH]:.4f})")


## 8. Analyze Predictions (Best Architecture)

In [ ]:
images, labels, preds, probs = collect_test_predictions(model, test_loader, DEVICE)
correct_idx, partial_idx, incorrect_idx = categorize_predictions(labels, preds)

show_prediction_examples(correct_idx,   images, labels, preds, "Fully Correct",     n=4)
show_prediction_examples(partial_idx,   images, labels, preds, "Partially Correct", n=4)
show_prediction_examples(incorrect_idx, images, labels, preds, "Fully Incorrect",   n=4)


In [ ]:
plot_per_class_metrics(labels, preds)
plot_confusion_matrices(labels, preds)
plot_prediction_heatmap(labels, preds)


## 9. Saliency Maps (Best Architecture)

In [ ]:
show_saliency_examples(correct_idx,   images, labels, preds, model, "Fully Correct",     n=5, device=DEVICE)
show_saliency_examples(partial_idx,   images, labels, preds, model, "Partially Correct", n=5, device=DEVICE)
show_saliency_examples(incorrect_idx, images, labels, preds, model, "Fully Incorrect",   n=5, device=DEVICE)
